[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/pctablet505/litert-demo/blob/main/litert_export_demo.ipynb)

# LiteRT Export Demo — Gemma 3 270M

Minimal, end-to-end demonstration of exporting a KerasHub `Gemma3CausalLM` to LiteRT (`.tflite`) from both the **TensorFlow** and **PyTorch** backends.

> **Note:** Switching `KERAS_BACKEND` between TF and Torch requires a **kernel restart**. This notebook keeps the two backends in separate sections so you can restart and run the relevant cell.

In [1]:
!pip install -q litert-torch
!pip install -q git+https://github.com/keras-team/keras.git
!pip install -q git+https://github.com/keras-team/keras-hub.git



   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 575.8/575.8 kB 20.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 419.3/419.3 kB 21.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 17.6/17.6 MB 59.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 117.6/117.6 MB 17.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.1/1.1 MB 43.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.2/3.2 MB 86.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 115.9/115.9 kB 8.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.4/56.4 kB 4.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 253.3/253.3 kB 18.2 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-genai 1.68.0 requires typing-extensions<5.0.0,>=4.14.0, but you have typing-extensions 4.12.2 which is 

In [1]:
PRESET = "hf://google/gemma-3-270m-it"
SEQ_LEN = 128

print("Preset:", PRESET)

Preset: hf://google/gemma-3-270m-it


## 1. TensorFlow Backend Export

The TF backend path works out of the box after `model.generate()` triggers `model.build()`. The exported model uses `TFLITE_BUILTINS + SELECT_TF_OPS`, so on Android you must link the Flex delegate:

```kotlin
implementation("org.tensorflow:tensorflow-lite-select-tf-ops:2.16.1")
```

**Run this section, then restart the kernel before switching to PyTorch.**

In [2]:
import os
os.environ["KERAS_BACKEND"] = "tensorflow"
os.environ["CUDA_VISIBLE_DEVICES"] = "-1"
os.environ["TF_ENABLE_ONEDNN_OPTS"] = "0"

import keras_hub

model = keras_hub.models.Gemma3CausalLM.from_preset(PRESET)
model.preprocessor.sequence_length = SEQ_LEN

# Build weights via generate()
out = model.generate(["What is Keras?"], max_length=16)
print("Generate output:", out)

# Export
model.export("gemma3_270m_tf.tflite", format="litert")
size_mb = round(os.path.getsize("gemma3_270m_tf.tflite") / 1e6, 1)
print("Size (MB):", size_mb)

config.json:   0%|          | 0.00/1.35k [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/536M [00:00<?, ?B/s]

tokenizer.model:   0%|          | 0.00/4.69M [00:00<?, ?B/s]

Generate output: ['What is Keras?\n\nKeras is a powerful and versatile Python library']
Saved artifact at 'gemma3_270m_tf.tflite'.
Size (MB): 1073.1


> **🔄 Restart the kernel now** (`Kernel → Restart`), then run the cell below to switch to the PyTorch backend.

## 2. PyTorch Backend Export

The PyTorch path requires an **explicit `input_signature`** because `get_input_signature()` returns `(None, None)` for the sequence dimension after build, causing the tracer to use `[1, 1]` instead of `[1, 128]`.

PyTorch export produces a cleaner graph (no Select TF ops) and is the preferred path for future StableHLO / AI-Edge-Torch investment.

In [7]:
import os
os.environ["KERAS_BACKEND"] = "torch"
os.environ["CUDA_VISIBLE_DEVICES"] = "-1"
os.environ["TF_ENABLE_ONEDNN_OPTS"] = "0"

import keras
import keras_hub
from keras import layers

model = keras_hub.models.Gemma3CausalLM.from_preset(PRESET)
model.preprocessor.sequence_length = SEQ_LEN

model.generate(inputs=["what is keras"])

# Explicit input signature for torch backend
input_signature = [{
    "token_ids": layers.InputSpec(dtype="int32", shape=(1, SEQ_LEN)),
    "padding_mask": layers.InputSpec(dtype="int32", shape=(1, SEQ_LEN)),
}]

model.export(
    "gemma3_270m_torch.tflite",
    format="litert",
    input_signature=input_signature,
)
size_mb = round(os.path.getsize("gemma3_270m_torch.tflite") / 1e6, 1)
print("Size (MB):", size_mb)


(00:00) [START] LiteRT-Torch Convert

(00:00) [START] LiteRT-Torch Convert > Torch Export: serving_default

(00:15) [START] LiteRT-Torch Convert > Torch Export: serving_default > ExportedProgram Run Decompositions

/usr/lib/python3.12/copyreg.py:99: FutureWarning: `isinstance(treespec, LeafSpec)` is deprecated, use `isinstance(treespec, TreeSpec) and treespec.is_leaf()` instead.
  return cls.__new__(cls, *args)


(00:34) [ DONE] LiteRT-Torch Convert > Torch Export: serving_default > ExportedProgram Run Decompositions (+00:19)

...

(01:54) [ DONE] LiteRT-Torch Convert > Run LiteRT Converter Passes (+00:31)

(01:54) [ DONE] LiteRT-Torch Convert (+01:54)

(00:00) [START] Write Model to gemma3_270m_torch.tflite

(00:02) [ DONE] Write Model to gemma3_270m_torch.tflite (+00:02)

Saved artifact at 'gemma3_270m_torch.tflite'.
Size (MB): 1073.5


> **🔄 Restart the kernel again** (back to TF backend) for the verification step, or use a fresh interpreter session.

## 3. Verify Inference (Torch Backend)

Load the PyTorch-exported `.tflite` with `ai_edge_litert.interpreter.Interpreter` and run the `serving_default` signature.

> The TF-backend `.tflite` requires the **Flex delegate** to run locally because it contains `tf.StridedSlice` Select TF ops. On Android this is handled by adding `org.tensorflow:tensorflow-lite-select-tf-ops` to `build.gradle`.

In [8]:
import os
os.environ["KERAS_BACKEND"] = "torch"
os.environ["CUDA_VISIBLE_DEVICES"] = "-1"
os.environ["TF_ENABLE_ONEDNN_OPTS"] = "0"

import numpy as np
from ai_edge_litert.interpreter import Interpreter

interp = Interpreter("gemma3_270m_torch.tflite")
runner = interp.get_signature_runner("serving_default")
print("Inputs:", list(runner.get_input_details().keys()))

result = runner(
    args_0=np.ones((1, 128), dtype=np.int32),
    args_1=np.ones((1, 128), dtype=np.int32),
)
print("Output shape:", result["output_0"].shape)
print("✅ LiteRT export verified.")

Inputs: ['args_0', 'args_1']
Output shape: (1, 128, 262144)
✅ LiteRT export verified.


> **🔄 Restart the kernel with TF backend** for the quantization step.

## 4. Post-Export Quantization — Smallest Size (INT4 Weights)

The FP32 `.tflite` from either backend is ~1,073 MB. For on-device deployment you typically need a much smaller model. The fastest path to smallest size is **weight-only INT4 quantization** via `ai-edge-quantizer`.

> **Why weight-only?** Activations stay in FP32, so accuracy degrades far less than full INT8 quantization, while weights shrink ~7–8×.

In [9]:
import os
os.environ["KERAS_BACKEND"] = "torch"
os.environ["CUDA_VISIBLE_DEVICES"] = "-1"
os.environ["TF_ENABLE_ONEDNN_OPTS"] = "0"

from ai_edge_quantizer import quantizer, recipe

# Quantize the torch-exported model (cleaner graph, no Flex ops)
input_path = "gemma3_270m_torch.tflite"
output_path = "gemma3_270m_torch_wi4afp32.tflite"

qt = quantizer.Quantizer(input_path)
qt.load_quantization_recipe(recipe.weight_only_wi4_afp32())

qt.quantize().export_model(output_path)

orig = os.path.getsize(input_path)
new = os.path.getsize(output_path)
print(f"Original:  {orig / 1e6:.1f} MB")
print(f"Quantized: {new / 1e6:.1f} MB")
print(f"Reduction: {orig / new:.1f}x")
print("✅ Quantized model ready for deployment.")

Applying Transformations to tensors:: 100%|██████████| 2131/2131 [00:00<00:00, 3915.37it/s] 


Model name: gemma3_270m_torch.tflite
Original model size: 1023.78 MiB
Quantized model size: 133.53 MiB
Quantization Ratio: 0.13 (7.7x smaller)
Total time: 3.04 s
Original:  1073.5 MB
Quantized: 140.0 MB
Reduction: 7.7x
✅ Quantized model ready for deployment.


## 5. Push to Android & Test

The quantized `.tflite` can be pushed to a device or emulator for testing with the raw Interpreter app (`gemmademo-android-app`).

For the **LiteRT-LM** app (`gemmademo-litertlm-android-app`), you need a `.litertlm` bundle. Either:
- Export directly with `format="litertlm"` (see `litertlm_export_demo.ipynb`), or
- Repackage the quantized TFLite + tokenizer + metadata into a `.litertlm` bundle using `litert_lm_builder`.

**ADB push to device:**

```bash
# For raw TFLite app
adb push gemma3_270m_torch_wi4afp32.tflite /sdcard/Android/data/com.example.gemmademo/files/

# For LiteRT-LM app
adb push gemma3_270m_it.litertlm /sdcard/Android/data/com.example.litertlmdemo/files/
```

> **Emulator note:** The LiteRT-LM app requires `arm64-v8a`. x86_64 emulators will fail unless the APK is rebuilt with `abiFilters += listOf("arm64-v8a", "x86_64")`.